# Fakeddit early-fusion baseline - free-tier Colab

Upload this project's Python files in the setup cell, then run the cells in order.

Drive must contain `MyDrive/fakeddit/tsv/` with the three multimodal TSV files.

In [26]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [27]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/fyp'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [28]:
import glob
import os
import shutil
import subprocess

PROJECT_DIR = '/content/drive/MyDrive/fyp'
CODE_DIR = '/content/code'
TSV_DIR = '/content/tsv'
DATASET_DIR = '/content/fakeddit_dataset'
os.makedirs(CODE_DIR, exist_ok=True)
os.makedirs(TSV_DIR, exist_ok=True)

project_files = [
    'dataset.py', 'download_data.py', 'evaluate.py',
    'extract_features.py', 'model.py', 'requirements.txt', 'train.py',
 ]
for filename in project_files:
    source = os.path.join(PROJECT_DIR, filename)
    if not os.path.exists(source):
        raise FileNotFoundError(f'Missing {filename} in {PROJECT_DIR}')
    shutil.copy2(source, os.path.join(CODE_DIR, filename))

required_tsv = {
    'multimodal_train.tsv',
    'multimodal_validate.tsv',
    'multimodal_test_public.tsv',
}
tsv_files = glob.glob(os.path.join(PROJECT_DIR, '**', 'multimodal_*.tsv'), recursive=True)
tsv_files += glob.glob('/content/drive/MyDrive/**/multimodal_*.tsv', recursive=True)
tsv_files = sorted(set(tsv_files))
found_names = {os.path.basename(path) for path in tsv_files}
missing = required_tsv - found_names

if missing:
    print('Fakeddit TSV files are not in Drive. Downloading the official dataset folder...')
    subprocess.run(['pip', '-q', 'install', 'gdown'], check=True)
    subprocess.run([
        'gdown', '--folder',
        'https://drive.google.com/drive/folders/1jU7qgDqU1je9Y0PMKJ_f31yXRo5uWGFm',
        '--output', DATASET_DIR, '--remaining-ok', '--continue',
    ], check=True)
    tsv_files = sorted(set(glob.glob(os.path.join(DATASET_DIR, '**', 'multimodal_*.tsv'), recursive=True)))
    found_names = {os.path.basename(path) for path in tsv_files}
    missing = required_tsv - found_names

if missing:
    raise FileNotFoundError(
        f'Missing required dataset files: {sorted(missing)}. '
        'The official dataset download did not contain the multimodal_only_samples splits.'
    )
for source in tsv_files:
    if os.path.basename(source) in required_tsv:
        shutil.copy2(source, os.path.join(TSV_DIR, os.path.basename(source)))

%cd /content/code
!pip -q install -r requirements.txt
print('Project files copied from:', PROJECT_DIR)
print('TSV files ready:', ', '.join(sorted(required_tsv)))

Fakeddit TSV files are not in Drive. Downloading the official dataset folder...
/content/code
Project files copied from: /content/drive/MyDrive/fyp
TSV files ready: multimodal_test_public.tsv, multimodal_train.tsv, multimodal_validate.tsv


## Features
Restores the feature cache from Drive if it exists; otherwise downloads images and extracts frozen features.

In [29]:
import os
import shutil
import subprocess
import tarfile

CACHE_DIR = '/content/cache'
CACHE_TAR = os.path.join(DRIVE, 'cache.tar')
REQUIRED_CACHE_FILES = [
    os.path.join(CACHE_DIR, 'meta.json'),
    os.path.join(CACHE_DIR, 'train', 'img.npy'),
    os.path.join(CACHE_DIR, 'validate', 'img.npy'),
    os.path.join(CACHE_DIR, 'test', 'img.npy'),
 ]

if os.path.exists(CACHE_TAR):
    shutil.rmtree(CACHE_DIR, ignore_errors=True)
    try:
        with tarfile.open(CACHE_TAR, 'r') as archive:
            archive.extractall('/content')
    except (tarfile.TarError, OSError):
        shutil.rmtree(CACHE_DIR, ignore_errors=True)

if not all(os.path.exists(path) for path in REQUIRED_CACHE_FILES):
    shutil.rmtree(CACHE_DIR, ignore_errors=True)
    subprocess.run([
        'python', 'download_data.py', '--data_dir', '/content/tsv',
        '--image_dir', '/content/images', '--limit', '60000', '10000', '10000',
    ], check=True)
    subprocess.run([
        'python', 'extract_features.py', '--data_dir', '/content/tsv',
        '--image_dir', '/content/images', '--cache_dir', CACHE_DIR,
    ], check=True)
    if not all(os.path.exists(path) for path in REQUIRED_CACHE_FILES):
        raise RuntimeError('Feature extraction finished without creating the complete cache.')
    with tarfile.open(CACHE_TAR, 'w') as archive:
        archive.add(CACHE_DIR, arcname='cache')

if not all(os.path.exists(path) for path in REQUIRED_CACHE_FILES):
    raise RuntimeError('Cache is incomplete. Rerun this cell after the feature extraction step succeeds.')
print('Feature cache ready:', CACHE_DIR)
!du -sh /content/cache

Feature cache ready: /content/cache
5.3G	/content/cache


## Train
Set `LABEL` to 2, 3, or 6, then run the cell.

In [30]:
import os
import subprocess

if not os.path.exists('/content/cache/train/img.npy'):
    raise FileNotFoundError('Feature cache is missing. Run cell 6 successfully before training.')

checkpoint_dir = os.path.join(DRIVE, 'checkpoints', f'{LABEL}way')
os.makedirs(checkpoint_dir, exist_ok=True)
subprocess.run([
    'python', 'train.py', '--cache_dir', '/content/cache',
    '--out_dir', checkpoint_dir, '--label', str(LABEL), '--epochs', '15', '--resume',
], check=True)

CompletedProcess(args=['python', 'train.py', '--cache_dir', '/content/cache', '--out_dir', '/content/drive/MyDrive/fyp/checkpoints/6way', '--label', '6', '--epochs', '15', '--resume'], returncode=0)

## Evaluate (all / image+text / image-only)

In [31]:
import os
import subprocess

checkpoint = os.path.join(
    DRIVE, 'checkpoints', f'{LABEL}way', f'best_{LABEL}way.pt'
 )
if not os.path.exists(checkpoint):
    raise FileNotFoundError(
        f'Checkpoint not found: {checkpoint}. Run the training cell successfully first.'
    )
if not os.path.exists('/content/cache/test/img.npy'):
    raise FileNotFoundError('Feature cache is missing. Run cell 6 successfully first.')

subprocess.run([
    'python', 'evaluate.py', '--ckpt', checkpoint,
    '--cache_dir', '/content/cache',
], check=True)

CompletedProcess(args=['python', 'evaluate.py', '--ckpt', '/content/drive/MyDrive/fyp/checkpoints/6way/best_6way.pt', '--cache_dir', '/content/cache'], returncode=0)